# 04 — Anomaly Detection (Isolation Forest + Z-Score)

**Goal:** Flag unusual trading days using two independent methods, then confirm anomalies only when both agree — reducing false positives dramatically.

| Method | What it detects | Weakness |
|---|---|---|
| **Isolation Forest** | Unusual *combinations* of return, volume, volatility, VIX | Can flag noisy micro-outliers |
| **Rolling Z-Score** | Single-day return that is >3σ from recent 30-day behaviour | Univariate — misses multi-feature patterns |

**Confirmed anomaly** = flagged by BOTH → high precision signal.

**DuckDB SQL** drives all summary tables, cross-year breakdowns, and regime analysis.

In [1]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import duckdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from pathlib import Path

plt.rcParams.update({
    'figure.dpi'      : 150,
    'font.family'     : 'DejaVu Sans',
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'axes.grid'       : True,
    'grid.alpha'      : 0.2,
    'grid.linestyle'  : ':',
})

DATA_DIR = Path('data')
OUT_DIR  = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

COLORS = {
    'AAPL': '#1565C0', 'AMZN': '#E65100', 'GLD' : '#2E7D32',
    'JPM' : '#C62828', 'SPY' : '#6A1B9A', 'XOM' : '#4E342E'
}

# Key market events — alternating y-heights to avoid label collisions
EVENT_DATES = [
    ('2020-03-16', 'COVID\nBlack Mon.', 0.97),
    ('2020-03-20', 'COVID\nBottom',     0.82),
    ('2022-01-24', 'Rate hike\nfear',   0.97),
    ('2022-09-13', 'CPI\nshock',        0.82),
    ('2023-03-10', 'SVB\ncollapse',     0.97),
]
print('Imports OK')

Imports OK


In [2]:
# ── 2. Load data ───────────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'market_data_indicators.csv', parse_dates=['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print(f'Loaded: {df.shape}  |  {df.Date.min().date()} → {df.Date.max().date()}')

Loaded: (12426, 25)  |  2018-01-02 → 2026-03-30


## Feature Engineering

Seven features feed Isolation Forest — four return/volatility features capture price dynamics; RSI captures momentum; VIX captures market-wide fear; VolChange captures unusual trading activity.

In [3]:
# ── 3. Feature engineering ────────────────────────────────────────────────
FEATURES = ['DailyReturn', 'LogReturn', 'VolChange', 'Vol5D', 'Vol30D', 'RSI14', 'VIX']

feat_frames = []
for ticker, grp in df.groupby('Ticker'):
    g = grp.copy().sort_values('Date').reset_index(drop=True)
    g['VolChange'] = g['Volume'].pct_change()
    g['Vol5D']     = g['DailyReturn'].rolling(5).std()  * np.sqrt(252) * 100
    g['Vol30D']    = g['DailyReturn'].rolling(30).std() * np.sqrt(252) * 100
    g['VIX']       = g['^VIX']
    feat_frames.append(g)

df = pd.concat(feat_frames, ignore_index=True).sort_values(['Ticker','Date']).reset_index(drop=True)
print(f'Features ready. Null counts:\n{df[FEATURES].isnull().sum().to_string()}')

Features ready. Null counts:
DailyReturn      6
LogReturn        6
VolChange        6
Vol5D           30
Vol30D         180
RSI14           84
VIX              0


## Train Isolation Forest — One Model Per Ticker

Separate models avoid the problem of AMZN's normal volatility looking like GLD's extreme event. `contamination=0.05` assumes 5% of days are anomalous.

In [4]:
# ── 4. Train Isolation Forest per ticker ──────────────────────────────────
model_frames = []

for ticker, grp in df.groupby('Ticker'):
    g = grp.copy().sort_values('Date').reset_index(drop=True)
    valid_mask = g[FEATURES].notna().all(axis=1)
    g_valid    = g[valid_mask].copy()
    if len(g_valid) < 50:
        continue

    X_scaled = StandardScaler().fit_transform(g_valid[FEATURES].values)
    iso       = IsolationForest(contamination=0.05, n_estimators=200,
                                random_state=42, n_jobs=-1)
    g_valid['IsoScore']      = iso.fit_predict(X_scaled)
    g_valid['IsAnomaly_ISO'] = (g_valid['IsoScore'] == -1).astype(int)
    g_valid['IsAnomaly_Z']   = g_valid['is_statistical_anomaly'].fillna(0).astype(int)
    g_valid['IsConfirmed']   = ((g_valid['IsAnomaly_ISO'] == 1) &
                                (g_valid['IsAnomaly_Z']   == 1)).astype(int)
    model_frames.append(g_valid)
    print(f'  {ticker}: ISO={g_valid.IsAnomaly_ISO.sum():>3}  '
          f'Z={g_valid.IsAnomaly_Z.sum():>3}  '
          f'Confirmed={g_valid.IsConfirmed.sum():>3}')

anomaly_df = pd.concat(model_frames, ignore_index=True).sort_values(['Ticker','Date']).reset_index(drop=True)
print(f'\nTotal confirmed anomalies: {anomaly_df.IsConfirmed.sum()}')

  AAPL: ISO=102  Z= 20  Confirmed= 14


  AMZN: ISO=102  Z= 17  Confirmed= 16


  GLD: ISO=102  Z= 14  Confirmed= 11


  JPM: ISO=102  Z= 16  Confirmed= 13


  SPY: ISO=102  Z= 23  Confirmed= 14


  XOM: ISO=102  Z= 10  Confirmed=  6

Total confirmed anomalies: 74


In [5]:
# ── 5. Save results ────────────────────────────────────────────────────────
OUT_CSV = DATA_DIR / 'anomaly_results.csv'
anomaly_df.to_csv(OUT_CSV, index=False)
print(f'Saved → {OUT_CSV}  ({OUT_CSV.stat().st_size/1024:.0f} KB)')

Saved → data\anomaly_results.csv  (4760 KB)


## SQL Block 1 — Anomaly Detection Summary

One query: counts of ISO flags, Z-score flags, confirmed anomalies, and confirmed rate % per ticker.

In [6]:
# ── 6. DuckDB — summary table ─────────────────────────────────────────────
con = duckdb.connect()
con.register('anomalies', anomaly_df)

summary = con.execute("""
    SELECT
        Ticker,
        COUNT(*)                                          AS total_days,
        SUM(IsAnomaly_ISO)                                AS iso_flags,
        SUM(IsAnomaly_Z)                                  AS z_flags,
        SUM(IsConfirmed)                                  AS confirmed,
        ROUND(SUM(IsConfirmed)*100.0 / COUNT(*), 2)       AS confirmed_rate_pct
    FROM anomalies
    GROUP BY Ticker
    ORDER BY confirmed_rate_pct DESC
""").df()

print('=== Anomaly Detection Summary ===')
print(summary.to_string(index=False))

=== Anomaly Detection Summary ===
Ticker  total_days  iso_flags  z_flags  confirmed  confirmed_rate_pct
  AMZN        2041      102.0     17.0       16.0                0.78
  AAPL        2041      102.0     20.0       14.0                0.69
   SPY        2041      102.0     23.0       14.0                0.69
   JPM        2041      102.0     16.0       13.0                0.64
   GLD        2041      102.0     14.0       11.0                0.54
   XOM        2041      102.0     10.0        6.0                0.29


## SQL Block 2 — Anomalies by Year

When did anomalies cluster? This reveals whether the model is capturing macro events (COVID 2020, rate-hike cycle 2022) or just noise.

In [7]:
# ── 7. DuckDB — confirmed anomalies by year ───────────────────────────────
by_year = con.execute("""
    SELECT
        YEAR(CAST(Date AS DATE))                          AS yr,
        SUM(IsConfirmed)                                  AS confirmed_anomalies,
        ROUND(AVG(CASE WHEN IsConfirmed=1
                       THEN DailyReturn END)*100, 2)      AS avg_ret_on_anomaly_pct,
        ROUND(AVG(CASE WHEN IsConfirmed=1
                       THEN VIX END), 1)                  AS avg_vix_on_anomaly
    FROM anomalies
    GROUP BY yr
    ORDER BY yr
""").df()

print('=== Confirmed Anomalies by Year ===')
print(by_year.to_string(index=False))

=== Confirmed Anomalies by Year ===
  yr  confirmed_anomalies  avg_ret_on_anomaly_pct  avg_vix_on_anomaly
2018                  6.0                   -0.01                21.9
2019                  4.0                    0.08                17.4
2020                 18.0                   -0.09                35.4
2021                  5.0                   -2.36                22.2
2022                  6.0                    0.77                27.0
2023                  5.0                   -0.47                19.5
2024                 11.0                    1.07                17.7
2025                 17.0                   -1.23                25.7
2026                  2.0                   -6.86                18.8


## SQL Block 3 — Anomaly Returns vs Normal Days

The key question: are anomaly days materially different from normal days? We compare average absolute return and average VIX.

In [8]:
# ── 8. DuckDB — anomaly vs normal day stats ───────────────────────────────
regime_stats = con.execute("""
    SELECT
        Ticker,
        IsConfirmed                                       AS anomaly_flag,
        COUNT(*)                                          AS days,
        ROUND(AVG(ABS(DailyReturn))*100, 3)               AS avg_abs_ret_pct,
        ROUND(STDDEV(DailyReturn)*SQRT(252)*100, 2)       AS ann_vol_pct,
        ROUND(AVG(VIX), 1)                                AS avg_vix
    FROM anomalies
    WHERE DailyReturn IS NOT NULL AND VIX IS NOT NULL
    GROUP BY Ticker, IsConfirmed
    ORDER BY Ticker, IsConfirmed DESC
""").df()

print('=== Anomaly Days vs Normal Days (IsConfirmed: 1=anomaly, 0=normal) ===')
print(regime_stats.to_string(index=False))

=== Anomaly Days vs Normal Days (IsConfirmed: 1=anomaly, 0=normal) ===
Ticker  anomaly_flag  days  avg_abs_ret_pct  ann_vol_pct  avg_vix
  AAPL             1    14            7.001       119.25     22.7
  AAPL             0  2027            1.303        29.08     19.8
  AMZN             1    16            8.745       146.93     21.7
  AMZN             0  2025            1.486        31.95     19.8
   GLD             1    11            4.529        59.79     21.8
   GLD             0  2030            0.714        15.53     19.8
   JPM             1    13            7.948       150.06     25.1
   JPM             0  2028            1.169        26.67     19.8
   SPY             1    14            4.420        73.16     31.4
   SPY             0  2027            0.775        18.19     19.8
   XOM             1     6            8.224       136.95     34.9
   XOM             0  2035            1.340        29.19     19.8


## SQL Block 4 — VIX Regime Buckets

Does the model catch more anomalies when VIX is elevated? Bucketing VIX into Low/Medium/High regimes with SQL answers this directly.

In [9]:
# ── 9. DuckDB — anomaly rate by VIX regime ────────────────────────────────
vix_regime = con.execute("""
    SELECT
        CASE
            WHEN VIX < 15  THEN '1_Low  (<15)'
            WHEN VIX < 25  THEN '2_Med  (15-25)'
            WHEN VIX < 40  THEN '3_High (25-40)'
            ELSE                '4_Extreme (>40)'
        END                                               AS vix_regime,
        COUNT(*)                                          AS total_days,
        SUM(IsConfirmed)                                  AS confirmed_anomalies,
        ROUND(SUM(IsConfirmed)*100.0 / COUNT(*), 2)       AS anomaly_rate_pct,
        ROUND(AVG(ABS(DailyReturn))*100, 3)               AS avg_abs_ret_pct
    FROM anomalies
    WHERE VIX IS NOT NULL AND DailyReturn IS NOT NULL
    GROUP BY vix_regime
    ORDER BY vix_regime
""").df()

print('=== Confirmed Anomaly Rate by VIX Regime ===')
print(vix_regime.to_string(index=False))
print('\nExpected: anomaly rate rises with VIX regime')

=== Confirmed Anomaly Rate by VIX Regime ===
     vix_regime  total_days  confirmed_anomalies  anomaly_rate_pct  avg_abs_ret_pct
   1_Low  (<15)        3090                  8.0              0.26            0.761
 2_Med  (15-25)        7014                 37.0              0.53            1.094
 3_High (25-40)        1908                 21.0              1.10            1.763
4_Extreme (>40)         234                  8.0              3.42            3.763

Expected: anomaly rate rises with VIX regime


## Anomaly Charts — Price + Confirmed Anomaly Markers

For each ticker:
- Price line with ticker colour
- **Red open circles** at confirmed anomaly dates
- **Orange dotted lines** at key market events with staggered labels (alternating heights to avoid overlap)
- Shaded band behind anomaly clusters

In [10]:
# ── 10. Anomaly charts per ticker ─────────────────────────────────────────
def plot_anomaly_chart(ticker_df, ticker, save_path):
    color = COLORS.get(ticker, '#444444')
    fig, (ax_price, ax_ret) = plt.subplots(
        2, 1, figsize=(15, 7), sharex=True,
        gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05}
    )

    # ── Top panel: price ──────────────────────────────────────────────
    ax_price.plot(ticker_df['Date'], ticker_df['Close'],
                  color=color, linewidth=1.5, label=f'{ticker} Close', zorder=3)

    # Confirmed anomaly circles
    anom = ticker_df[ticker_df['IsConfirmed'] == 1]
    ax_price.scatter(
        anom['Date'], anom['Close'],
        facecolors='none', edgecolors='red',
        s=100, linewidths=1.8, zorder=5, label=f'Confirmed Anomaly (n={len(anom)})'
    )

    # Event reference lines with staggered labels
    ymin_p, ymax_p = ticker_df['Close'].min(), ticker_df['Close'].max()
    y_range = ymax_p - ymin_p
    for date_str, label, y_frac in EVENT_DATES:
        xd = pd.Timestamp(date_str)
        if not (ticker_df['Date'].min() <= xd <= ticker_df['Date'].max()):
            continue
        ax_price.axvline(x=xd, color='#F57C00', linestyle=':', linewidth=1.4,
                         alpha=0.9, zorder=2)
        ax_price.text(
            xd, ymin_p + y_range * y_frac, label,
            color='#E65100', fontsize=7.5, ha='center', va='top',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='#FFF3E0',
                      edgecolor='#FF9800', alpha=0.85)
        )

    ax_price.set_title(f'{ticker} — Price & Confirmed Anomalies',
                       fontsize=13, fontweight='bold', pad=10)
    ax_price.set_ylabel('Close Price ($)', fontsize=10)
    ax_price.legend(loc='upper left', fontsize=9, framealpha=0.9)

    # ── Bottom panel: daily return bar ───────────────────────────────
    ret = ticker_df['DailyReturn'].fillna(0)
    bar_colors = ['#C62828' if r < 0 else '#2E7D32' for r in ret]
    ax_ret.bar(ticker_df['Date'], ret * 100, color=bar_colors,
               alpha=0.7, width=1.5, zorder=2)
    ax_ret.axhline(0, color='black', linewidth=0.6)
    ax_ret.set_ylabel('Daily\nReturn (%)', fontsize=9)

    # Mark anomaly return bars in darker red
    anom_ret = ticker_df[ticker_df['IsConfirmed'] == 1]
    ax_ret.bar(anom_ret['Date'], anom_ret['DailyReturn'] * 100,
               color='red', alpha=1.0, width=1.5, zorder=4)

    # Shared x-axis
    for ax in (ax_price, ax_ret):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(ax_ret.xaxis.get_majorticklabels(), rotation=0, ha='center', fontsize=9)

    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved → {save_path}')

for ticker in sorted(anomaly_df['Ticker'].unique()):
    sub = anomaly_df[anomaly_df['Ticker'] == ticker].copy()
    plot_anomaly_chart(sub, ticker, OUT_DIR / f'anomaly_{ticker}.png')

print('\nAll anomaly charts saved.')

  Saved → outputs\anomaly_AAPL.png


  Saved → outputs\anomaly_AMZN.png


  Saved → outputs\anomaly_GLD.png


  Saved → outputs\anomaly_JPM.png


  Saved → outputs\anomaly_SPY.png


  Saved → outputs\anomaly_XOM.png

All anomaly charts saved.


In [11]:
print('\n✓ 04_anomaly complete')
for f in sorted(OUT_DIR.glob('anomaly_*.png')):
    print(f'  → {f.name}  ({f.stat().st_size/1024:.1f} KB)')


✓ 04_anomaly complete
  → anomaly_AAPL.png  (148.2 KB)
  → anomaly_AMZN.png  (165.6 KB)
  → anomaly_GLD.png  (137.4 KB)
  → anomaly_JPM.png  (142.1 KB)
  → anomaly_SPY.png  (146.3 KB)
  → anomaly_XOM.png  (144.6 KB)
